<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/fabiobento/dnn-course-2026-1/blob/main/C1_M4_Lab_2_debugging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

# Depuração, Inspeção e Modularização de Modelos

Até agora, você se concentrou em construir e treinar modelos.
Mas, no mundo real, a sua primeira tentativa em um modelo raramente funciona perfeitamente.
Muitas vezes, você encontrará mensagens de erro enigmáticas sobre formatos incompatíveis de tensores (*mismatched tensor shapes*), ou pior, o seu modelo será executado sem erros, mas não conseguirá produzir resultados significativos.

É aqui que a **depuração (*debugging*), inspeção e modularização** se tornam habilidades essenciais.
Neste laboratório, você assumirá o papel de um investigador de modelos.
Você começará com uma Rede Neural Convolucional (CNN) quebrada e usará técnicas sistemáticas de depuração para encontrar e corrigir o bug. Em seguida, você aprenderá como refatorar o seu código para clareza e reutilização e, finalmente, você dissecará um modelo complexo pré-treinado para entender o seu funcionamento interno.

Neste laboratório, você irá:

* **Depurar** uma CNN quebrada inserindo instruções de impressão (*print statements*) na passagem direta (*forward pass*) para identificar e corrigir uma incompatibilidade crítica no formato de um tensor.
* **Refatorar** o modelo corrigido usando `nn.Sequential` para criar uma arquitetura mais limpa, mais modular e menos propensa a erros.
* **Inspecionar** as estatísticas de ativação do seu modelo para realizar uma verificação de sanidade (*sanity check*) em busca de problemas como gradientes explodindo ou desaparecendo (*exploding/vanishing gradients*).
* **Explorar** a arquitetura de um modelo complexo pré-existente (`SqueezeNet`) para contar suas camadas e analisar a sua distribuição de parâmetros.

## Importação de Bibliotecas

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models import SqueezeNet

In [ ]:
import helper_utils

## Carregamento de Dados

Para depurar e inspecionar um modelo de forma eficaz, primeiro você precisará de um conjunto de dados para trabalhar. O objetivo deste laboratório é praticar um fluxo de trabalho de depuração e inspeção de ponta a ponta (*end-to-end*), portanto, você usará um conjunto de dados simples que permite focar na arquitetura do modelo em vez de em um pré-processamento de dados complexo.

Para este propósito, você usará o conjunto de dados Fashion MNIST, que consiste em imagens em tons de cinza de peças de vestuário e serve como uma referência (*benchmark*) direta para tarefas de classificação de imagens. Você começará carregando o conjunto de dados usando a biblioteca `torchvision` do PyTorch e, em seguida, criará um `DataLoader` para lidar com os dados de forma eficiente em lotes (*batches*) durante o treinamento e a avaliação.

In [ ]:
dataset = helper_utils.get_dataset()

transform = transforms.ToTensor()
dataset.transform = transform

In [ ]:
batch_size = 64
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

In [ ]:
img_batch, label_batch = next(iter(dataloader))
print("Batch shape:", img_batch.shape)  # Should be [batch_size, 1, 28, 28]

## Depuração através do *Forward Pass*

Ao começar a trabalhar com um novo modelo, é comum encontrar erros.
Esses erros podem ocorrer por vários motivos, como formatos incorretos de tensores (*tensor shapes*), operações incompatíveis ou valores inesperados.
Às vezes, o modelo pode ser executado sem erros, mas produzir saídas incorretas.

Nesta seção, você explorará como depurar um modelo PyTorch examinando a sua passagem direta (*forward pass*).

### Uma primeira exploração do modelo

Agora é a hora de explorar o modelo.
Este modelo é uma rede simples com:

* um bloco convolucional: consistindo de uma camada convolucional, uma função de ativação ReLU e uma camada de agrupamento máximo (*max pooling*),
* um bloco totalmente conectado (*fully connected*): consistindo de uma camada linear, uma função de ativação ReLU e uma camada linear final que produz as pontuações das classes.

Primeiro, você instanciará o modelo e tentará executar uma passagem direta (*forward pass*) com um lote (*batch*) do carregador de dados (*dataloader*).
Para obter uma saída mais limpa em caso de erros, você usará `try/except` para capturar quaisquer exceções que possam surgir durante a passagem direta.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Convolutional Block
        self.conv = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully Connected Block
        # For Fashion MNIST: input images are 28x28,
        # after conv+pool: 32x14x14
        self.fc1 = nn.Linear(32 * 14 * 14, 128)
        self.relu_fc = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)  # 10 classes for Fashion MNIST

    def forward(self, x):
        x = self.pool(self.relu(self.conv(x)))
        x = self.relu_fc(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
simple_cnn = SimpleCNN()

try:
    output = simple_cnn(img_batch)  
except Exception as e:
    print(f"\033[91mError during forward pass: {e}\033[0m")

De fato, o modelo conforme fornecido contém alguns erros que requerem depuração.
A mensagem fornecida pelo PyTorch quando ocorre um erro às vezes pode ser um pouco enigmática (*cryptic*).
Ela descreve que duas matrizes (`mat1` e `mat2`) não podem ser multiplicadas e fornece os seus formatos (*shapes*).
Isso indica que há uma incompatibilidade nas dimensões dos tensores sendo multiplicados, o que é um problema comum em implementações de redes neurais.

No entanto, a mensagem de erro não especifica **por que e onde** no modelo o erro ocorre.
É aí que o método `forward` do modelo entra em ação.
A natureza de grafo dinâmico do PyTorch permite que você insira instruções de impressão (*print statements*) ou use ferramentas de depuração para inspecionar os valores e os formatos dos tensores em vários pontos no método `forward`.

Você definirá uma nova classe que herda do modelo original e substitui (*overrides*) o método `forward` para incluir instruções de impressão que exibem o formato do tensor após cada camada.
Uma primeira tentativa pode ser separar explicitamente as camadas no método `forward` e, para cada camada:

* imprimir o formato do tensor antes da camada (formato de entrada),
* imprimir o formato de alguns *parâmetros da camada* (por exemplo, pesos e vieses - *weights and biases*),
* imprimir o formato do tensor de *ativação* após a camada (formato de saída), que será a entrada para a próxima camada.

Você agora pode executar a passagem direta (*forward pass*) novamente e observar os formatos impressos para identificar onde a incompatibilidade ocorre.

In [ ]:
class SimpleCNNDebug(SimpleCNN):
    def __init__(self):
        super().__init__()
        # The super().__init__() call above properly initializes all layers from SimpleCNN
        # No need to redefine the layers here

    def forward(self, x):
        print("Input shape:", x.shape)
        print(
            " (Layer components) Conv layer parameters (weights, biases):",
            self.conv.weight.shape,
            self.conv.bias.shape,
        )
        x_conv = self.relu(self.conv(x))

        print("===")

        print("(Activation) After convolution and ReLU:", x_conv.shape)
        x_pool = self.pool(x_conv)
        print("(Activation) After pooling:", x_pool.shape)

        print(
            "(Layer components) Linear layer fc1 parameters (weights, biases):",
            self.fc1.weight.shape,
            self.fc1.bias.shape,
        )

        x_fc1 = self.relu_fc(self.fc1(x_pool))

        print("===")

        print("(Activation) After fc1 and ReLU:", x_fc1.shape)

        print(
            "(Layer components) Linear layer fc2 parameters (weights, biases):",
            self.fc2.weight.shape,
            self.fc2.bias.shape,
        )
        x = self.fc2(x_fc1)

        print("===")

        print("(Activation) After fc2 (output):", x.shape)
        return x

In [ ]:
simple_cnn_debug = SimpleCNNDebug()

try:
    output_debug = simple_cnn_debug(img_batch)  
except Exception as e:
    print(f"\033[91mError during forward pass in debug model: {e}\033[0m")

Esta já é uma saída mais limpa. Você já pode ver que todas as camadas do bloco convolucional estão funcionando bem, e os formatos (*shapes*) estão como o esperado (`batch_size=64` e `out_channels=32`).

**O erro ocorre no bloco totalmente conectado**, especificamente na primeira camada linear: `x_pool` tem o formato `[64, 32, 14, 14]`, mas a camada linear espera uma entrada com o formato `[64, 6272]` (sua matriz de pesos tem o formato `[128, 6272]`).

Como a camada linear `fc1` espera uma entrada 2D com formato `[batch_size, input_features]`, o `x_pool` é achatado para um tensor 2D com o formato `[64*32*14, 14]` antes de ser passado para `fc1`. Este não é o formato pretendido, e isso leva ao erro de incompatibilidade de dimensões.

Uma vez que você tenha identificado o problema, poderá corrigi-lo adicionando uma operação de achatamento (*flattening*) antes da primeira camada linear no método `forward`.

In [ ]:
class SimpleCNNFixed(SimpleCNN):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        print("Input shape:", x.shape)
        print(
            " (Neuron components) Conv layer parameters (weights, biases):",
            self.conv.weight.shape,
            self.conv.bias.shape,
        )
        x_conv = self.relu(self.conv(x))

        print("===")

        print("(Activation) After convolution and ReLU:", x_conv.shape)
        x_pool = self.pool(x_conv)
        print("(Activation) After pooling:", x_pool.shape)

        x_flattened = torch.flatten(
            x_pool, start_dim=1
        )  # Flatten all dimensions except batch
        print("(Activation) After flattening:", x_flattened.shape)

        print(
            "(Neuron components) Linear layer fc1 parameters (weights, biases):",
            self.fc1.weight.shape,
            self.fc1.bias.shape,
        )

        x_fc1 = self.relu_fc(self.fc1(x_flattened))

        print("===")

        print("(Activation) After fc1 and ReLU:", x_fc1.shape)

        print(
            "(Neuron components) Linear layer fc2 parameters (weights, biases):",
            self.fc2.weight.shape,
            self.fc2.bias.shape,
        )
        x = self.fc2(x_fc1)

        print("===")

        print("(Activation) After fc2 (output):", x.shape)
        return x

In [ ]:
# Fixed version
simple_cnn_fixed = SimpleCNNFixed()

output = simple_cnn_fixed(img_batch)

O problema agora está corrigido e o modelo é executado sem erros! Você pode ver que os formatos dos tensores estão como o esperado após cada camada, e a saída final tem o formato correto de `[64, 10]`, correspondendo ao tamanho do lote (*batch size*) e ao número de classes.

Uma vez que o modelo está rodando sem erros, você pode avançar para a próxima seção para refatorar o modelo usando `nn.Sequential` e obter uma implementação mais limpa e modular.

## `nn.Sequential` para Modularização

O modelo agora está funcionando corretamente, mas o método `forward` é bastante verboso e repetitivo.
Para tornar o código mais limpo e modular, você pode usar o `nn.Sequential` para definir os blocos convolucionais e totalmente conectados.

Desta forma, você obtém várias vantagens:

* **Modularidade**: Cada bloco é definido como um módulo separado, tornando-o mais fácil de entender e modificar.
* **Reutilização**: Você pode facilmente reutilizar os blocos em outros modelos ou experimentos.
* **Código Mais Limpo**: O método `forward` se torna muito mais simples, pois ele só precisa chamar os blocos sequencialmente.
* **Menos Propenso a Erros**: Ao definir os blocos em um único lugar, você reduz as chances de cometer erros ao implementar o método `forward`.

In [ ]:
class SimpleCNN2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        # Convolutional Block
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Fully Connected Block
        # For Fashion MNIST: input images are 28x28,
        # after conv+pool: 32x14x14
        flattened_size = 32 * 14 * 14
        self.fc_block = nn.Sequential(
            nn.Linear(flattened_size, 128),
            nn.ReLU(),
            nn.Linear(128, 10),  # 10 classes for Fashion MNIST
        )

    def forward(self, x):
        x = self.conv_block(x)
        x = torch.flatten(x, start_dim=1)  # Flatten all dimensions except batch
        x = self.fc_block(x)
        return x

In [ ]:
simple_cnn_seq = SimpleCNN2Seq()
output = simple_cnn_seq(img_batch)

print("Output shape from sequential model:", output.shape)

### Inspeção Estatística da Inicialização

Uma verificação comum ao inspecionar um modelo é observar as estatísticas de algumas ativações para garantir que elas estejam dentro de um intervalo razoável.

In [ ]:
class SimpleCNN2SeqDebug(SimpleCNN2Seq):
    def __init__(self):
        super().__init__()
        # The super().__init__() call above properly initializes all layers from SimpleCNN2Seq
        # No need to redefine the layers here

    def get_statistics(self, activation):
        mean = activation.mean().item()
        std = activation.std().item()
        min_val = activation.min().item()
        max_val = activation.max().item()

        print(f" Mean: {mean}")
        print(f" Std: {std}")
        print(f" Min: {min_val}")
        print(f" Max: {max_val}")
        return mean, std, min_val, max_val

    def forward(self, x):
        features = self.conv_block(x)
        x = torch.flatten(features, start_dim=1)  # Flatten all dimensions except batch

        print("After conv_block, the activation statistics are:")
        self.get_statistics(features)

        x = self.fc_block(x)
        print("After fc_block, the activation statistics are:")
        self.get_statistics(x)
        return x

In [ ]:
simple_cnn_seq_debug = SimpleCNN2SeqDebug()

for idx, (img_batch, _) in enumerate(dataloader):
    if idx < 5:
        print(f"=== Batch {idx} ===")
        output_debug = simple_cnn_seq_debug(img_batch)

Esta é uma verificação de sanidade (*sanity check*) para garantir que o modelo seja inicializado corretamente e que as ativações não estejam explodindo ou desaparecendo (*exploding or vanishing*).
*Esses problemas podem levar a um mau desempenho no treinamento ou a problemas de convergência.*

## Inspeção do Modelo

Com o modelo anterior funcionando corretamente, agora você inspecionará um modelo complexo pré-existente do `torchvision.models`, como a `SqueezeNet`.

Nesta seção, você utilizará os utilitários de inspeção fornecidos pelo PyTorch para explorar a arquitetura, as camadas e os parâmetros do modelo.
Essas técnicas de inspeção são fundamentais para uma depuração eficaz e para realizar modificações fundamentadas nos designs das suas redes neurais.

### Visão geral da arquitetura

In [ ]:
# Load SqueezeNet model
complex_model = SqueezeNet()

print(complex_model)

Para modelos complexos, imprimir toda a arquitetura do modelo pode ser excessivo. Em vez disso, você pode utilizar `named_children()` e `children()` para iterar pelos blocos de nível superior do modelo.

In [ ]:
# Iterate through the main blocks
for name, block in complex_model.named_children():
    print(f"Block {name} has a total of {len(list(block.children()))} layers:")
    
    # List all children layers in the block
    for idx, layer in enumerate(block.children()):
        # Check if the layer is terminal (no children) or not
        if len(list(layer.children())) == 0:
            print(f"\t {idx} - Layer {layer}")
        # If the layer has children, it's a sub-block, then print only the number of children and its name
        else:
            layer_name = layer._get_name()  # More user-friendly name
            print(f"\t {idx} - Sub-block {layer_name} with {len(list(layer.children()))} layers")            

Isso fornece uma visão geral mais limpa da estrutura do modelo, permitindo que você se concentre nos componentes principais sem se perder nos detalhes de cada camada individual.
Agora, você examinará mais de perto um dos módulos `Fire` para ver a sua estrutura interna.

Para isso, você pode usar `modules()` para iterar por todas as camadas e submódulos do modelo.

In [ ]:
first_fire_module = complex_model.features[3]

for idx, module in enumerate(first_fire_module.modules()):
    # Avoid printing the top-level module itself
    if idx > 0 :
        print(module)

Agora a arquitetura do modelo está impressa de forma organizada, mostrando os componentes principais e as suas configurações.

Você agora pode realizar algumas inspeções específicas, como contar o número de tipos específicos de camadas ou calcular o número total de parâmetros do modelo.

### Inspeção Detalhada

Você agora contará quantas camadas `Conv2d` existem no modelo.

In [ ]:
type_layer = nn.Conv2d

selected_layers = [layer for layer in complex_model.modules() if isinstance(layer, type_layer)]

print(f"Number of {type_layer.__name__} layers: {len(selected_layers)}")

Você agora contará o número total de parâmetros no modelo.
Isso lhe dá uma ideia da complexidade e da capacidade do modelo.

In [ ]:
# total number of parameters in the model
total_params = sum(p.numel() for p in complex_model.parameters())
print(f"Total number of parameters in the model: {total_params}")

Agora, você pode ir além inspecionando os parâmetros de cada camada terminal (camadas sem filhos) no modelo.
Para cada camada terminal, você imprimirá o seu nome e o número total de parâmetros que ela contém.
Isso ajuda a identificar quais camadas contribuem mais para a contagem de parâmetros do modelo e pode ser útil para a otimização do modelo, poda (*pruning*) ou para entender onde reside a capacidade do modelo.

In [ ]:
counting_params = {}

# For each terminal layer print its number of parameters
for layer in complex_model.named_modules():
    n_children = len(list(layer[1].children()))
    if n_children == 0:  # Terminal layer
        layer_name = layer[0]
        n_parameters = sum(p.numel() for p in layer[1].parameters())
        counting_params[layer_name] = n_parameters
        print(f"Layer {layer_name} has {n_parameters} parameters")

# Plotting the distribution of parameters per layer
helper_utils.plot_counting(counting_params)

# Conclusão

Você agora depurou, refatorou e inspecionou modelos PyTorch com sucesso.
Neste laboratório, você viu em primeira mão que a passagem direta (*forward pass*) de um modelo não é uma caixa preta e que, adicionando estrategicamente instruções de impressão (*print statements*), você pode diagnosticar e resolver erros comuns, mas frustrantes, como incompatibilidades de formato (*shape mismatches*).

Você foi além de simplesmente escrever o código do modelo e agora pode torná-lo mais robusto e legível agrupando camadas em blocos lógicos com o **`nn.Sequential`**.
Essa prática de modularização torna as suas arquiteturas mais fáceis de entender, reutilizar e adaptar.
Você também aprendeu a realizar verificações de sanidade (*sanity checks*) essenciais inspecionando as estatísticas de ativação e a explorar sistematicamente qualquer modelo PyTorch, por mais complexo que seja, usando utilitários de inspeção como `.modules()` e `.named_children()`.

Com essas habilidades fundamentais de depuração e inspeção, você está bem preparado para desafios mais avançados.